# Post 3 — Evaluation (§7)

Assembles point/probabilistic metrics, the paired Wilcoxon test (LightGBM vs TabPFN-TS), and the rankings-flip chart. The harness runs with whatever per-series parquets exist: with only LightGBM present it reports the LightGBM side and notes the pair as **pending**; adding the TabPFN path to `PER_SERIES_SOURCES` activates the comparison with no other change.

## 0. Control panel

In [ ]:
# ---- Control panel (evaluation §7) ----------------------------------------
CONFIG_NAME = "default.yaml"
OUTPUT_SUBDIR = "evaluation"
USE_CORE_SLICE = True
# Per-series metric parquets written by model notebooks, keyed by approach label.
# Add the TabPFN path here once §6.2 produces it.
PER_SERIES_SOURCES = {
    "lgbm_point": "lgbm_engineered",   # dir under results/ holding per_series_mae_*.parquet
}
QUICK = False
RANDOM_SEED = 42

## 1. Imports

In [ ]:
import sys, time, json, copy, platform, subprocess
from pathlib import Path

# Make `src` importable whether the notebook runs from notebooks/ or repo root.
REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "src" / "demand").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.demand.config import load_config, resolve_path
from src.demand.data.load import load_raw_data, build_panel
from src.demand.data.splits import (
    build_main_splits, build_core_slice, family_regime, regime_horizon, regime_costs,
)
from src.demand.data.supervised_frame import build_supervised_frame
from src.demand.models import lgbm_point, lgbm_quantile
from src.demand.models.lgbm_base import train_lgbm

def git_commit():
    try:
        return subprocess.check_output(["git", "rev-parse", "--short", "HEAD"],
                                       stderr=subprocess.DEVNULL).decode().strip()
    except Exception:
        return "no-git"

## 2. Load data + scope

In [ ]:
cfg = copy.deepcopy(load_config(CONFIG_NAME))
cfg["seed"] = RANDOM_SEED
splits = build_main_splits(cfg)

t0 = time.time()
raw = load_raw_data(cfg)
panel = build_panel(raw, include_test=False)
print(f"loaded panel in {time.time()-t0:.1f}s | rows={len(panel):,} | "
      f"series={panel[['store_nbr','family']].drop_duplicates().shape[0]:,}")

if USE_CORE_SLICE:
    series_filter = build_core_slice(panel, raw.stores, cfg, write_csv=False)
    print(f"core slice: {len(series_filter)} series, "
          f"{series_filter['family'].nunique()} families")
else:
    series_filter = None
    print("scope: full panel")

results_dir = resolve_path(cfg, "results_dir")
figures_dir = resolve_path(cfg, "figures_dir")
out_dir = results_dir / OUTPUT_SUBDIR
out_dir.mkdir(parents=True, exist_ok=True)
print(f"git commit: {git_commit()}  ->  output: {out_dir}")

## 3. Per-series metrics + paired Wilcoxon (§7.3)

In [ ]:
from src.demand.evaluation.statistical_tests import run_pairwise_tests
from src.demand.evaluation import plots

# 1) Gather per-series regime MAE for each available approach.
per_series = {}
for approach, subdir in PER_SERIES_SOURCES.items():
    d = results_dir / subdir
    frames = [pd.read_parquet(p) for p in d.glob("per_series_mae_*.parquet")]
    if not frames:
        # Fall back to the flat artifact name from the quick run.
        flat = d / "per_series_mae.parquet"
        if flat.exists():
            frames = [pd.read_parquet(flat)]
    if frames:
        per_series[approach] = pd.concat(frames, ignore_index=True)
        print(f"{approach}: {len(per_series[approach]):,} paired rows")
    else:
        print(f"{approach}: no per_series_mae parquet found under {d}")

# 2) Paired Wilcoxon (§7.3) for every model pair that is actually present.
approaches = list(per_series)
pairs = [(a, b) for i, a in enumerate(approaches) for b in approaches[i+1:]]
if pairs:
    wilcoxon = run_pairwise_tests(per_series, pairs, value_col="mae")
    display(wilcoxon)
    wilcoxon.to_parquet(results_dir / "wilcoxon_tests.parquet", index=False)
else:
    print("Only one approach present — Wilcoxon needs the TabPFN-TS leg (§6.2) "
          "to form a pair. Structure is ready; re-run once it exists.")

## 4. Rankings flip by metric (§7.2)

In [ ]:
# §7.2 — rankings-flip chart. Pull each approach's mean value per metric from its
# metrics_all.parquet (point metrics) so we can spot rank reversals across metrics.
metric_rows = []
for approach, subdir in PER_SERIES_SOURCES.items():
    d = results_dir / subdir
    m = None
    for cand in [*d.glob("metrics_all.parquet"), d / "metrics.parquet"]:
        if cand.exists():
            m = pd.read_parquet(cand); break
    if m is None:
        continue
    for metric in ("mae", "rmse", "smape", "bias"):
        if metric in m.columns:
            metric_rows.append({"approach": approach, "metric": metric,
                               "value": float(m[metric].mean())})
metrics_long = pd.DataFrame(metric_rows)
enough = (not metrics_long.empty) and metrics_long["approach"].nunique() >= 2
if enough:
    plots.ranking_flip(metrics_long, figures_dir / OUTPUT_SUBDIR)
else:
    print("ranking-flip chart needs >=2 approaches; pending TabPFN-TS leg.")
display(metrics_long)

from src.demand.reporting.summary_tables import build_summary_tables
from src.demand.reporting.narrative_bullets import build_narrative_bullets
print("summary   ->", build_summary_tables(cfg))
print("narrative ->", build_narrative_bullets(cfg))